[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bobleesj/quantem.widget/blob/main/notebooks/showdiffraction/showdiffraction_simple.ipynb)
# ShowDiffraction — Quick Demo
Interactive d-spacing measurement for 4D-STEM diffraction patterns.
Dual-panel: diffraction pattern (left) + BF virtual image (right).
Click on the diffraction pattern to measure d-spacings.
Spots table shows position, d-spacing (Å), |g| (1/Å), and intensity.

In [1]:
# Install in Google Colab
try:
    import google.colab
    !pip install -q -i https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ quantem-widget
except ImportError:
    pass  # Not in Colab, skip

In [2]:
try:
    %load_ext autoreload
    %autoreload 2
    %env ANYWIDGET_HMR=1
except Exception:
    pass  # autoreload unavailable (Colab Python 3.12+)

env: ANYWIDGET_HMR=1


In [3]:
import torch
import numpy as np
import quantem.widget
from quantem.widget import ShowDiffraction, profile

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


def make_4dstem_nanoparticle(scan_rows=32, scan_cols=32, det_rows=128, det_cols=128):
    """Synthetic 4D-STEM: crystalline nanoparticle on amorphous substrate.

    The BF virtual image shows a bright circular nanoparticle (~10 nm)
    on a darker amorphous background. The diffraction pattern changes
    depending on scan position: crystalline Bragg spots inside the
    particle, diffuse rings on the substrate.
    """
    dr = torch.arange(det_rows, device=device, dtype=torch.float32)
    dc = torch.arange(det_cols, device=device, dtype=torch.float32)
    rr, cc = torch.meshgrid(dr, dc, indexing="ij")
    cr, cc0 = det_rows / 2, det_cols / 2
    center_dist = ((rr - cr) ** 2 + (cc - cc0) ** 2).sqrt()

    # === Crystalline pattern (inside nanoparticle) ===
    # BF disk
    bf_radius = 10.0
    bf = (center_dist < bf_radius).float() * (
        1.0 + 0.15 * torch.cos(center_dist * 0.8)
    )
    # 6 first-order Bragg spots (hexagonal)
    bragg = torch.zeros(det_rows, det_cols, device=device)
    for k in range(6):
        angle = k * torch.pi / 3
        sr = cr + 22.0 * torch.sin(torch.tensor(angle, device=device))
        sc = cc0 + 22.0 * torch.cos(torch.tensor(angle, device=device))
        d2 = (rr - sr) ** 2 + (cc - sc) ** 2
        bragg += 0.5 * torch.exp(-d2 / (2 * 2.5 ** 2))
    # 6 second-order spots
    for k in range(6):
        angle = k * torch.pi / 3 + torch.pi / 6
        sr = cr + 38.0 * torch.sin(torch.tensor(angle, device=device))
        sc = cc0 + 38.0 * torch.cos(torch.tensor(angle, device=device))
        d2 = (rr - sr) ** 2 + (cc - sc) ** 2
        bragg += 0.15 * torch.exp(-d2 / (2 * 2.0 ** 2))
    crystal_pattern = bf + bragg  # (det, det)

    # === Amorphous pattern (substrate) ===
    # Diffuse ring at ~20 px radius + weaker BF disk
    amorphous_ring = 0.3 * torch.exp(-((center_dist - 20) ** 2) / (2 * 5 ** 2))
    amorphous_bf = (center_dist < bf_radius).float() * 0.4
    amorphous_bg = 0.03 * torch.exp(-center_dist / 50)
    amorphous_pattern = amorphous_bf + amorphous_ring + amorphous_bg  # (det, det)

    # === Spatial map: nanoparticle mask ===
    si = torch.arange(scan_rows, device=device, dtype=torch.float32)
    sj = torch.arange(scan_cols, device=device, dtype=torch.float32)
    si_grid, sj_grid = torch.meshgrid(si, sj, indexing="ij")
    # Circular nanoparticle, offset from center
    np_center_r, np_center_c = scan_rows * 0.45, scan_cols * 0.55
    np_radius = min(scan_rows, scan_cols) * 0.25
    scan_dist = ((si_grid - np_center_r) ** 2 + (sj_grid - np_center_c) ** 2).sqrt()
    # Smooth edge (tanh transition over ~2 px)
    crystal_weight = 0.5 * (1.0 - torch.tanh((scan_dist - np_radius) / 1.5))  # (scan, scan)

    # Thickness variation within particle (dome profile)
    thickness = torch.clamp(1.0 - (scan_dist / np_radius) ** 2, min=0.0)
    thickness_mod = 0.8 + 0.4 * thickness  # range [0.8, 1.2]

    # === Combine: crystal inside particle, amorphous outside ===
    # (scan, scan, 1, 1) * (1, 1, det, det)
    data = (
        crystal_weight[:, :, None, None] * crystal_pattern[None, None]
        * thickness_mod[:, :, None, None]
        + (1.0 - crystal_weight[:, :, None, None]) * amorphous_pattern[None, None]
    )

    # Poisson shot noise
    counts = 300
    if device.type == "mps":
        data = torch.poisson(data.clamp(min=0).cpu() * counts) / counts
    else:
        data = torch.poisson(data.clamp(min=0) * counts) / counts

    return data.cpu().numpy()


data = make_4dstem_nanoparticle()
print(f"Shape: {data.shape}, dtype: {data.dtype}")
print(f"Range: [{data.min():.3f}, {data.max():.3f}]")
print(f"quantem.widget {quantem.widget.__version__}")
profile()

Using device: cpu


Shape: (32, 32, 128, 128), dtype: float32
Range: [0.000, 1.587]
quantem.widget 0.0.14
quantem.widget  0.0.14
quantem         0.1.8
Python          3.14.4 (CPython)
NumPy           2.4.4
PyTorch         2.11.0+cu130
Compute         cpu
System RAM      7.7 GB (1.6 GB available)
VRAM            N/A
Disk            41.6 GB free / 926.4 GB
Jupyter         Lab 4.5.6
anywidget       0.10.0
Platform        Linux 6.12.76-linuxkit aarch64


## Basic Usage
The BF virtual image (right) shows a bright nanoparticle on an amorphous substrate.
Click inside or outside the particle to see how the diffraction pattern changes:
sharp Bragg spots on the crystal, diffuse rings on the amorphous region.

In [4]:
# k_pixel_size calibrates reciprocal space: d = 1 / (|g| * k_pixel_size)
w = ShowDiffraction(
    data,
    k_pixel_size=0.025,   # 1/Å per pixel
    pixel_size=0.5,       # Å per scan pixel (real space)
    title="Au Nanoparticle on a-C",
)
w

  to cpu: 0.03s (67.1 MB)


ShowDiffraction(shape=(32, 32, 128, 128), sampling=(0.5 Å, 0.025 1/Å), pos=(16, 16), title='Au Nanoparticle on a-C')

## Programmatic Spot Measurement
Navigate to the nanoparticle center, then measure the first-order Bragg spots.

In [5]:
# Move to the center of the nanoparticle
w.position = (14, 18)

# Measure first-order Bragg spots
w.add_spot(row=42, col=64)   # top spot
w.add_spot(row=64, col=83)   # right spot
w.add_spot(row=64, col=45)   # left spot

# Print d-spacings
for spot in w.spots:
    d = f"{spot['d_spacing']:.2f} Å" if spot['d_spacing'] else f"{spot['r_pixels']:.1f} px"
    g = f"{spot['g_magnitude']:.4f} 1/Å" if spot['g_magnitude'] else "--"
    print(f"  Spot #{spot['id']}: ({spot['row']:.0f}, {spot['col']:.0f}) d={d}, |g|={g}")

  Spot #1: (42, 64) d=1.82 Å, |g|=0.5505 1/Å
  Spot #2: (64, 83) d=2.11 Å, |g|=0.4742 1/Å
  Spot #3: (64, 45) d=2.10 Å, |g|=0.4758 1/Å


## Snap to Peak
Enable snap-to-peak so clicks lock onto the nearest intensity maximum.
Click approximately near a spot — it snaps to the exact peak.

In [6]:
w2 = ShowDiffraction(
    data,
    k_pixel_size=0.025,
    pixel_size=0.5,
    title="Snap to Peak",
    snap_enabled=True,
    snap_radius=5,
)
w2.position = (14, 18)  # on the nanoparticle

# Click near a Bragg spot (off by a few pixels) — snaps to the peak
w2.add_spot(row=44, col=66)  # slightly off → snaps to actual peak
print(f"Clicked (44, 66) → snapped to ({w2.spots[0]['row']:.0f}, {w2.spots[0]['col']:.0f})")
w2

  to cpu: 0.02s (67.1 MB)
Clicked (44, 66) → snapped to (44, 71)


ShowDiffraction(shape=(32, 32, 128, 128), sampling=(0.5 Å, 0.025 1/Å), pos=(14, 18), spots=1, title='Snap to Peak')

## Inspect Widget State

In [7]:
w.summary()

Au Nanoparticle on a-C
════════════════════════════════
Scan:     32×32 (0.50 Å/px)
Detector: 128×128 (0.0250 1/Å/px)
Position: (14, 18)
Center:   (64.0, 64.0)  BF r=23.8 px
Spots:    3
  #1 (42.0, 64.0) d=1.817 Å
  #2 (64.0, 83.0) d=2.109 Å
  #3 (64.0, 45.0) d=2.102 Å
Display:  inferno | log


In [8]:
# Save and reload state (preserves spots, display settings, calibration)
w.save("showdiffraction_state.json")

w3 = ShowDiffraction(data, k_pixel_size=0.025, state="showdiffraction_state.json", verbose=False)
print(f"Restored {len(w3.spots)} spots")
w3

Restored 3 spots


ShowDiffraction(shape=(32, 32, 128, 128), sampling=(0.5 Å, 0.025 1/Å), pos=(14, 18), spots=3, title='Au Nanoparticle on a-C')